# U-Net 二段階学習（Google Colab）

1. **パッチ化** … 合成 1,000 + Guitar-TECHS
2. **Stage 1** … 合成データで学習
3. **Stage 2** … Guitar-TECHS でファインチューニング
4. **push** … `checkpoints/stage2/unet_last.pt` を GitHub へ

事前: `REPO_URL` と `GITHUB_TOKEN`（PAT）を設定

In [ ]:
REPO_URL = "https://github.com/Kakeru0307/colab_train.git"
GITHUB_TOKEN = ""  # ← PAT（repo 権限）

import os
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

In [ ]:
import os
from pathlib import Path

if not Path('train.py').exists():
    !git clone {REPO_URL} repo
    %cd repo/colab_train
else:
    print('既にリポジトリ内です:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
SYNTHETIC_COUNT = 1000
SYNTHETIC_SEED = 42

!python prepare_all.py --synthetic-count {SYNTHETIC_COUNT} --synthetic-seed {SYNTHETIC_SEED}

In [ ]:
STAGE1_EPOCHS = 20
STAGE2_EPOCHS = 20
BATCH_SIZE = 16

!python train.py \
  --pairs-dir data/pairs/synthetic \
  --epochs {STAGE1_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --checkpoint-dir checkpoints/stage1 \
  --pos-weight 10

In [ ]:
!python train.py \
  --pairs-dir data/pairs/guitar-techs \
  --epochs {STAGE2_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr 1e-5 \
  --checkpoint-dir checkpoints/stage2 \
  --resume checkpoints/stage1/unet_last.pt \
  --pos-weight 10

In [ ]:
!git config user.email "colab@users.noreply.github.com"
!git config user.name "Colab Train"
!python scripts/push_checkpoint.py \
  --ckpt checkpoints/stage2/unet_last.pt \
  --message "Stage1+2 train synthetic+techs"